In [1]:
import pymmcore_plus
from pymmcore_plus import CMMCorePlus
from useq import MDAEvent
from useq import MDASequence
from pathlib import Path
import useq
import numpy as np
import tifffile as tiff
import datetime
from VoiceCoil import DAQ_VC
_instance_MDA = None
mmc = None
DAQ = None

In [2]:
class MDA:
    @classmethod
    def instance(cls) -> 'MDA':
        """Return the global instance of `MDA`.
        """
        global _instance_MDA
        if _instance_MDA is None:
            _instance_MDA = cls()
        return _instance_MDA
        
    def __init__(
        self,
        config_path: str = r"\Users\Hannah\Desktop\configuration\PVCAM_only.cfg"
    ):
        
        # Set the first instance of this class as the global singleton
        global _instance_MDA
        if _instance_MDA is None:
            _instance_MDA = self
        global mmc
        if mmc is not None:
            mmc.unloadAllDevices()
        if mmc is None:
            mmc = CMMCorePlus.instance()
        mmc.enableDebugLog(True)
        # Load the correct configuration file. NB! If a new configuration file is created, change the path to the new configuration file
        mmc.loadSystemConfiguration(config_path)
        mmc.setAutoShutter(False)
        #settings needed to set up the DAQ
        global DAQ
        if DAQ is None:
            DAQ = DAQ_VC()
        #Setup the default Camera settings for 1 fps rolling shutter.
        self._exposure  = 2.44
        self._scan_width = 8
        self._scan_direction = 'UP'
        self._trigger  = 'Edge Trigger'
        self._Port = 'Dynamic Range'
        self._save = False

        
    def setup(
        self
    ):
        today =datetime.datetime.now()   # Get date
        datestring = today.strftime("%Y%m%d")  # Date to the desired string format
        Path(datestring).mkdir(parents=True, exist_ok=True)   # Create folder
        self.tif_writer = tiff.TiffWriter("zstack.tif")
        try:
            mmc.mda.events.frameReady.disconnect(on_frame)
        except Exception:
            pass
        @mmc.mda.events.frameReady.connect
        def on_frame(self, image: np.ndarray, event: useq.MDAEvent, meta: pymmcore_plus.metadata.FrameMetaV1):
            # do what you want with the data
            print(
                f"received frame:temp{event.metadata['idx']}" 
            )
            if self._save:
                tiff.imwrite(f"{Path(datestring)}\\temp{event.metadata['idx']}.tif", image, photometric='minisblack')
    @property
    def save(self) -> bool:
        """Turn saving on and off
        
        Returns
        -------
        scan_type: str
            scan type. One of "2d", "mirror", "projection", or "stage"
        """
        
        return getattr(self,"_save",None)
    @save.setter
    def scan_type(self, value: bool):
        """Set the scan type.
        
        Parameters
        ----------
        value: str
            scan type. One of "2d", "mirror", "projection", or "stage
        """
        
        if not hasattr(self, "_save") or self._save is None:
            self._save = value
        else:
            self._save = value
            
    @property
    def exposure(self) -> float:
        """Turn saving on and off
        
        Returns
        -------
        scan_type: str
            scan type. One of "2d", "mirror", "projection", or "stage"
        """
        
        return getattr(self,"_exposure",None)
    @exposure.setter
    def exposure(self, value: float):
        """Set the scan type.
        
        Parameters
        ----------
        value: str
            scan type. One of "2d", "mirror", "projection", or "stage
        """
        
        if not hasattr(self, "_exposure") or self._exposure is None:
            self._exposure = value
        else:
            self._exposure = value
    
    @property
    def scan_width(self) -> int:
        """Turn saving on and off
        
        Returns
        -------
        scan_type: str
            scan type. One of "2d", "mirror", "projection", or "stage"
        """
        
        return getattr(self,"_scan_width",None)
    @scan_width.setter
    def scan_type(self, value: int):
        """Set the scan type.
        
        Parameters
        ----------
        value: str
            scan type. One of "2d", "mirror", "projection", or "stage
        """
        
        if not hasattr(self, "_scan-width") or self._scan_width is None:
            self._scan_width = value
        else:
            self._scan_width = value
    def Sequence(
        self,
        Step_size_z: float = 0.2,
        X_tiles: int = 0,
        Y_tiles: int = 0,
        )
    def close(self):
        mmc.unloadAllDevices()

In [3]:
test = MDA()

In [4]:
test.setup()

In [6]:
test.close()

In [5]:
DAQ.program_daq_waveforms()

<CMMCorePlus at 0x272a04c2250 with 1 devices>
